In [ ]:
## R code for computing Polygenic Risk Scores (PRSs) using pre-trained models downloaded from the PGS Catalog.
## The PRS is calculated using genetic data from the All of Us (AoU) Research Program and implemented with PLINK.

In [ ]:
#install.packages("openxlsx")
#install.packages('R.utils')
## load in R packages

library(openxlsx)
library(R.utils)
library(tidyverse)
library(lubridate)  
library(bigrquery)  
library(data.table)

In [ ]:
## get basic information

## get the bucket name
my_bucket <- Sys.getenv('WORKSPACE_BUCKET')

## my_bucket: gs://fc-secure-XXXXXX

## get the BigQuery curated dataset for the current workspace context
CDR <- Sys.getenv('WORKSPACE_CDR')

## CDR: fc-aou-cdr-prod-ct.C2022Q4R13


In [ ]:
## EFO trait IDs from the PGS Catalog
## https://www.pgscatalog.org/browse/traits/
## Examples:
## Body Mass Index (BMI): EFO_0004340
## Low-Density Lipoprotein (LDL): EFO_0004611
## High-Density Lipoprotein (HDL): EFO_0004612

EFO_list = c("EFO_0004340", "EFO_0004611", "EFO_0004612")

In [ ]:
## copy pgscatalog metadata to local VM
system("gsutil cp -n gs://fc-secure-XXXXXX/pgs_all_metadata.xlsx ./")

## copy plink2 to local VM
system("gsutil cp -n gs://fc-secure-XXXXXX/data/plink2 ./")

## make plink2 executable
system("chmod 751 ./plink2")

In [ ]:
## open the pgs metadata
df <- read.xlsx("pgs_all_metadata.xlsx", sheet = "Scores")
for(efo in EFO_list){
    if(file.exists(paste0("./keep_scores_",efo,'.txt'))){
        cat('PGS score exists: pass', '\n')
        next}
    #check if efo exist
  keep.scores <- df[df[,5] == efo,]
    
  if(nrow(keep.scores) < 0)next 
  #download the (harmonized hg38) score files
  for(i in 1:nrow(keep.scores)) {
    system(paste0("wget -nc https://ftp.ebi.ac.uk/pub/databases/spot/pgs/scores/",keep.scores[i,'Polygenic.Score.(PGS).ID'],"/ScoringFiles/Harmonized/",keep.scores[i,1],"_hmPOS_GRCh38.txt.gz"))
  }
  #move plink files to local VM
  system("gsutil -m cp -n gs://fc-secure-XXXXXX/data/* ./", intern=T)
  #update the plink bim file to have CHR:POS as ID
  #since a lot of pgscatalog files don't have RSID
  for(i in c("afr", "amr", "eas", "eur")){
    system(paste0("./plink2 --set-all-var-ids '@:#:$1:$2' --bfile ", 
                  i, " --make-just-bim --out ", i))
  }

  #make scoring files for plink
  #update ID
  bim1 <- data.table::fread("afr.bim")
  bim2 <- data.table::fread("amr.bim")
  bim3 <- data.table::fread("eas.bim")
  bim4 <- data.table::fread("eur.bim")
  bim <- bind_rows(list(bim1, bim2, bim3, bim4))
  bim <- bim[!duplicated(bim$V2),]
  keep.scores$overlap <- NA
  keep.scores$overlap2 <- NA
    
      for(i in 1:nrow(keep.scores)) {
    tmp <- data.table::fread(paste0(keep.scores[i,'Polygenic.Score.(PGS).ID'],"_hmPOS_GRCh38.txt.gz"), data.table=F)
    if(!all(c("hm_chr", "hm_pos", "effect_allele", "other_allele") %in% names(tmp))) {
      keep.scores[i,1] <- NA
    } else {
      tmp$rsID <- with(tmp, paste(hm_chr, hm_pos, apply(cbind(effect_allele, other_allele), 1, function(x) paste(sort(x), collapse = ":")), sep = ":"))
      tmp <- tmp[!duplicated(tmp$rsID),]
      overlap <- sum(tmp$rsID %in% bim$V2)/nrow(tmp)
      overlap2 <- sum(bim$V2 %in% tmp$rsID)/nrow(bim)
      keep.scores$overlap[i] <- overlap   
      keep.scores$overlap2[i] <- overlap2
      if(overlap == 0 & overlap2 == 0) {
        keep.scores[i,1] <- NA
      } else {
        tmp <- tmp[!duplicated(tmp$rsID),]
        write.table(tmp[,c("rsID", "effect_allele", "effect_weight")], 
                    file = paste0(keep.scores$`Polygenic.Score.(PGS).ID`[i], ".txt"), 
                    col.names = F, sep = '\t', row.names=F, quote=F)
      }
    }
  }
                                                    
keep.scores <- keep.scores[!is.na(keep.scores[,1]),]
write.table(keep.scores, paste0("keep_scores_",efo,".txt"), quote = F, row.names = F)
system(paste0("gsutil cp ./keep_scores_",efo,".txt gs://fc-secure-XXXXXX/data/", efo, "/"))

#make a scorelist file holding all the pgs names for plink
  out <- paste0(keep.scores$`Polygenic.Score.(PGS).ID`, ".txt")
  write.table(out, paste0("scorelist_", efo), col.names=F, row.names=F, quote=F)
  out <- system(paste0("cat scorelist_", efo), intern=T)
  cat('Keep scoring')

  #calc the sores
  for(i in c("afr", "amr", "eas", "eur")) {
    system(paste0("plink2 --bfile ", i, 
                  " --score-list ", paste0("scorelist_", efo),
                  " --read-freq ", paste0(i, ".afreq"),
                  " --out ", i, "_", efo))
  }
  cat('PGS scoring completed')
  #rename the columns in the file to match the PGScatalog ID
  for(i in c("afr", "amr", "eas", "eur")) {
    out <- readr::read_tsv(paste0(i, "_", efo,".sscore"))
    scores <- read.table(paste0("scorelist_", efo))
    names(out) <- c("FID", "IID", gsub(".txt", "", scores$V1))
    write.table(out, paste0(i, "_", efo,"_final.sscore"), quote = F, row.names=F)
  }
  cat('PGS writing completed')
  
  #move the score files to bucket
  for(i in c("afr", "amr", "eas", "eur")) {
    system(paste0("gsutil cp ./", i, "_", efo,"_final.sscore gs://fc-secure-XXXXXX/data/",
             efo, "/"))
  }
  #remove the scores to save space
  file.remove(list.files(pattern=".sscore"))
}